<a href="https://colab.research.google.com/github/DebDuttz/Training-D2/blob/main/Groq_tooling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata

GROQ_API_KEY  = userdata.get("GROQ_API_KEY")
SUPABASE_URL  = userdata.get("SUPABASE_URL")
SUPABASE_KEY  = userdata.get("SUPABASE_KEY")

print("✅ Secrets loaded" if all([GROQ_API_KEY, SUPABASE_URL, SUPABASE_KEY]) else "❌ Something is missing")

✅ Secrets loaded


In [2]:
!pip install -q groq supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.5 MB/s eta 0:00:00


In [3]:
from supabase import create_client

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Quick sanity check — read 3 rows
test = supabase.table("employees").select("*").limit(3).execute()
for row in test.data:
    print(row["name"], "—", row["role"])

Priya Sharma — Senior Developer
Rahul Verma — Developer
Aisha Khan — Data Scientist


In [4]:
def get_employees(department: str = None, city: str = None) -> list:
    """Fetch employees from the database, optionally filtered.

    Args:
        department: e.g. "Engineering", "Data", "Sales", "Design"
        city:       e.g. "Bangalore", "Pune", "Delhi", "Mumbai"
    Returns:
        A list of employee records (dicts).
    """
    query = supabase.table("employees").select(
        "name, department, role, city, salary"
    )
    if department:
        query = query.eq("department", department)
    if city:
        query = query.eq("city", city)

    result = query.execute()
    return result.data

# Test the tool directly (no AI yet — just checking it works)
print(get_employees(department="Data"))

[{'name': 'Aisha Khan', 'department': 'Data', 'role': 'Data Scientist', 'city': 'Pune', 'salary': 1600000}, {'name': 'Vikram Singh', 'department': 'Data', 'role': 'Data Analyst', 'city': 'Pune', 'salary': 900000}, {'name': 'Divya Nair', 'department': 'Data', 'role': 'ML Engineer', 'city': 'Bangalore', 'salary': 1900000}]


In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_employees",
            "description": (
                "Get the list of company employees. Can filter by department "
                "and/or city. Use this whenever the user asks about employees, "
                "staff, teams, who works where, salaries, or headcount."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "department": {
                        "type": "string",
                        "description": "Department name, e.g. Engineering, Data, Sales, Design",
                    },
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. Bangalore, Pune, Delhi, Mumbai",
                    },
                },
                "required": [],
            },
        },
    }
]

In [6]:
import json
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"   # a solid Groq model with reliable tool calling

# Maps the tool name (a string) to the real Python function
available_functions = {"get_employees": get_employees}

def ask_bot(question: str) -> str:
    messages = [
        {"role": "system", "content":
            "You are a helpful HR assistant. Use the provided tools to answer "
            "questions about employees. Answer in clear, friendly sentences."},
        {"role": "user", "content": question},
    ]

    # --- STEP 1: Ask Groq. Does it want to use a tool? ---
    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",   # let the model decide
    )
    msg = response.choices[0].message

    # --- STEP 2: If it requested tool(s), run them ---
    if msg.tool_calls:
        messages.append(msg)   # record the model's request

        for call in msg.tool_calls:
            fn_name = call.function.name
            fn_args = json.loads(call.function.arguments)
            print(f"   🔧 calling {fn_name}({fn_args})")

            # Run OUR real function against the real database
            result = available_functions[fn_name](**fn_args)

            # Hand the result back to the model
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": fn_name,
                "content": json.dumps(result),
            })

        # --- STEP 3: Ask Groq again → it writes the final human answer ---
        final = groq_client.chat.completions.create(model=MODEL, messages=messages)
        return final.choices[0].message.content

    # No tool needed — the model answered directly
    return msg.content

In [7]:
print(ask_bot("Who works in the Data team?"))

   🔧 calling get_employees({'department': 'Data'})
The Data team consists of Aisha Khan, a Data Scientist, Vikram Singh, a Data Analyst, and Divya Nair, an ML Engineer.


In [8]:
print("🤖 HR Bot ready! Ask me about employees. Type 'quit' to exit.\n")

while True:
    question = input("You: ")
    if question.strip().lower() in {"quit", "exit", "q"}:
        print("👋 Bye!")
        break
    if not question.strip():
        continue

    answer = ask_bot(question)
    print(f"Bot: {answer}\n")

🤖 HR Bot ready! Ask me about employees. Type 'quit' to exit.

You: highest salary among them
   🔧 calling get_employees({})
Bot: The highest salary among the employees is 1,900,000, which is held by Divya Nair, an ML Engineer in the Data department, located in Bangalore.

You: who is late jonnie
Bot: I'm not aware of any information about a person named Late Jonnie. Could you please provide more context or clarify who Late Jonnie is? I'll do my best to help.

You: who joined late
   🔧 calling get_employees({})
Bot: Based on the data, the following employees joined late: 

1. Rohit Kumar 
2. Vikram Singh

You: which department earns the most
   🔧 calling get_employees({})
Bot: The department that earns the most is Engineering, with an average salary of 1500000. However, the Data department has the highest individual earner, with Divya Nair earning a salary of 1900000.



KeyboardInterrupt: Interrupted by user